# 00 · Setup

Shared configuration loaded by every Spark notebook with `%run ./00_setup.ipynb`:
project paths and a local **SparkSession with Delta Lake**.

You don't need to run this notebook by itself.

In [13]:
import os
from pathlib import Path

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

# Spark 3.5 supports Java 17, so I used temurin on my Fedora
JAVA_17 = Path("/usr/lib/jvm/temurin-17-jdk")
os.environ["JAVA_HOME"] = str(JAVA_17)

# Notebooks run from the project root folder
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
assert DATA_DIR.exists(), f"'data/' not found in {BASE_DIR}. Open the notebooks from the project root."

LANDING_CSV = DATA_DIR / "landing" / "enem_2020" / "MICRODADOS_ENEM_2020.csv"
BRONZE_TABLE = str(DATA_DIR / "bronze" / "enem_raw")
SILVER_TABLE = str(DATA_DIR / "silver" / "enem_cleaned")
GOLD_DIR = DATA_DIR / "gold"

In [14]:
builder = (
    SparkSession.builder.appName("ETL-ENEM-Medallion")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "4")  # small value for a single local machine
    # Delta JARs are downloaded once from Maven and cached inside the project (.ivy2/),
    # so later runs work offline
    .config("spark.jars.ivy", str(BASE_DIR / ".ivy2"))
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

# configure_spark_with_delta_pip adds the Delta JAR matching the installed delta-spark version
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} ready")

Spark 3.5.3 ready
